In [1]:
# folder with text files
folder_with_text = './text'

# max text length
max_length=1024

# latent space z size
internal_dim=32

# training batch size
batch_size=128

# where to save model checkpoints
checkpoint_path = 'runs/test-autoregressive-gd2-padmask'

# number of epochs to train
num_epochs=500

# Create tokenizer

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from typing import List, Dict
import os
from kemsekov_torch.text_tools import SimpleTokenizer


txt_files = [[os.path.join(fdir,f) for f in files if f.endswith(".txt")] for fdir,_,files in os.walk(folder_with_text)]
txt_files = [b for a in txt_files for b in a]
txt_lines = [open(v).read() for v in txt_files]

tokenizer = SimpleTokenizer(txt_lines,lowercase=True,unknown_symbols_placeholder=' ')
torch.jit.script(tokenizer).save("tokenizer.pt")

test_str="This is my TEST string! Раз!"
inds=tokenizer.encode(test_str)
print(test_str)
print(inds)
print(tokenizer.decode(inds))

Text length analysis
text lines	 79295
line chars mean	 78.267
line chars std	 180.905
0.05 quantile	 0.0
0.95 quantile	 341.0
0.995 quantile	 711.0
This is my TEST string! Раз!
tensor([54, 42, 43, 53,  1, 43, 53,  1, 47, 59,  1, 54, 39, 53, 54,  1, 53, 54,
        52, 43, 48, 41,  2,  1,  1,  1,  1,  2])
this is my test string!    !


/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Define Dataset

In [3]:
import math
from kemsekov_torch.train import split_dataset
import torch
from kemsekov_torch.text_tools import TokenDataset

txt_split = [b for v in txt_lines if len(v)>30 for b in v.split('\n')]
dataset = TokenDataset(
    tokenizer,
    txt_split,
    pad_token=tokenizer.unknown_symbols_placeholder,
    batch_size=batch_size,
    fixed_length=max_length
)

# split dataset into train and test
train_dataset,test_dataset,train_loader, test_loader = split_dataset(
    dataset,
    test_size=0.05,
    batch_size=batch_size,
    random_state=None,
    # bin_by_size=True,
    num_workers=1,
)

Train items 36358
Test items 1914


In [4]:
import random
ind = random.randint(0,len(train_dataset)-1)
inds = dataset[ind]

print("Text length",len(inds))
skip_text = tokenizer.decode(inds).strip()
print(skip_text)

for t in train_loader: break
print("batch size sample",t.shape)

Text length 1024
harry sat in seething silence, glaring at dumbledore. what was going on? did this mean that dumbledore had indeed ordered snape to find out what malfoy was doing, in which case he had already heard everything harry had just told him from snape? or was he really worried by what he had heard, but pretending not to be?


batch size sample torch.Size([128, 1024])


# Define Model

In [5]:
from autoregressive import AutoregressiveChar

model = AutoregressiveChar(tokenizer.vocab_size,256,layers=2,mlp_factor=2,impl='gd2')
print(model.params_count())
[c.shape for c in model(t[:,:128])]

2010884


[torch.Size([128, 128, 256]), torch.Size([128, 128, 80])]

# Training

In [6]:
from kemsekov_torch.train import train
from kemsekov_torch.metrics import f1_score
from accelerate.utils import TorchDynamoPlugin
from torchmetrics.classification import MulticlassF1Score

# Initialize the metric object
f1_metric = MulticlassF1Score(num_classes=tokenizer.vocab_size, average='macro').cuda()

CE = torch.nn.CrossEntropyLoss()


def get_pad_mask(next_t: torch.Tensor, pad_token: int) -> torch.Tensor:
    """
    Finds the first occurrence of 3 sequential pad tokens per batch row 
    and returns a flattened boolean mask of shape [BATCH * seqlen].
    Elements at and after the 3 pads are set to False.
    
    We use this thing to compute loss on non-padded part of batch
    """
    is_pad = (next_t == pad_token)
    
    # Check 3 sequential tokens using slicing
    sequential_3_pads = is_pad[:, :-2] & is_pad[:, 1:-1] & is_pad[:, 2:]
    
    # Find the first index along dim=1 where this happens per batch item
    has_3_pads = sequential_3_pads.any(dim=-1, keepdim=True)
    first_pad_idx = torch.argmax(sequential_3_pads.int(), dim=-1, keepdim=True)
    
    # Create index grid to build the mask
    seq_indices = torch.arange(next_t.shape[1], device=next_t.device).unsqueeze(0)
    
    # Retain elements before the 3-pad boundary
    mask = torch.ones_like(next_t, dtype=torch.bool)
    mask = torch.where(has_3_pads, seq_indices < first_pad_idx, mask)
    
    return mask

def compute_loss_and_metric(model,batch):
    prev_t = batch[:,:-1]
    next_t = batch[:,1:]
    activations,logits = model(prev_t)
    mask = get_pad_mask(next_t, pad_token=dataset.pad_token[0]).flatten()
    
    logits=logits.view(-1,logits.shape[-1])[mask]
    next_t=next_t.flatten()[mask]
    
    loss = CE(logits,next_t)
    f1 = f1_metric(logits,next_t)
    return loss,{
        'f1':f1
    }

_ = train(
    model,
    train_loader,
    test_loader,
    compute_loss_and_metric,
    checkpoint_path,
    # f'{checkpoint_path}/last',
    gradient_clipping_max_norm=1,
    accelerate_args=dict(
        mixed_precision='fp16',
        dynamo_plugin = TorchDynamoPlugin(
            backend="inductor",
            mode="default",
            fullgraph=False,
            dynamic=True          # Enables torch.compile(dynamic=True)
        )
    ),
    save_on_metric_improve=['f1'],
    num_epochs=100,
    checkpoints_count=1,
    # default_lr=0.01
)

/home/bochkarev/Programs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using dir runs/test-autoregressive-gd2-padmask
Using default fused AdamW optimizer
Using default CosineAnelingScheduler
Total model parameters 2.01 M
Using device cuda

Epoch 1/100


train 0: 100%|██████████| 284/284 [01:37<00:00,  2.93it/s, f1=0.3113, loss=1.3557]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.71065 | 1.39655 |
|  f1  | 0.2223  | 0.2895  |
+------+---------+---------+
saved epoch-1

Epoch 2/100


train 0: 100%|██████████| 284/284 [01:12<00:00,  3.91it/s, f1=0.3506, loss=1.2262]


...

+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 0.95205 | 1.09105 |
|  f1  | 0.4769  | 0.4301  |
+------+---------+---------+

Epoch 24/100


train 0: 100%|██████████| 284/284 [01:15<00:00,  3.74it/s, f1=0.5055, loss=0.9043]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 0.94739 | 1.09174 |
|  f1  | 0.4789  | 0.4251  |
+------+---------+---------+

Epoch 25/100


train 0:  90%|█████████ | 256/284 [01:06<00:07,  3.87it/s, f1=0.4711, loss=0.8978]


Interrupt training
Empty cuda cache complete


In [7]:
!python sample_gd2.py --prompt "The Hanged Man, the village pub" --to_generate 1024

/home/bochkarev/Programs/AutoregressiveChar/sample_gd2.py:16: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  tokenizer=torch.load("tokenizer.pt",weights_only=False)
/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
Loading last checkpoint at epoch 21
the hanged man, the village pub of the fingers on the right halfway down the stairs with the rest of the stairs, said, 'and something in the mirror of the wizards have defeated me.' he thought that he was not containing a day at the top box of chaser’s distractions with the shrieking hand with his hair and shaking, and harry thought that he had been a reliving was now signed on the path with something. the light was not only the light of all he was wearing a long yellow and the